# 🎯 [MOV01_M03] 구간별 통과 용량 기반 1차 수송 지체 예측 Baseline 알고리즘 구현 (정답 솔루션)
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV01/mov01_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 3) |
| **미션 ID** | MOV01_M03 |
| **미션 주제** | 구간별 통과 용량 기반 1차 수송 지체 예측 Baseline 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수송 소요를 데이터로 예측해 군수 수송계획을 세우는 일은 구간별 통과 용량과 병목을 함께 보아야 정확해진다. 본 프로젝트는 수송 소요예측에서 예측 대상 정의, 시계열·피처 구성, 기준모델 비교, 설명가능성, 업무 임계값, 실적 대조를 익히도록 교육용 가상 노선망 자료를 코드로 만들어 구간별 수송 소요와 지체를 예측한다. 학습자는 병목 데이터 정의, 지체 전처리, BPR 기준 모델, 최대유량 기반 병목 식별과 피처 모델, 우회 소요·유류 산출과 앙상블 모델, 혼잡도 대시보드와 요인 설명, 작전군수관 승인(HITL)과 선형계획 최적화, 3개 구간 동시 차단 스트레스 시험, AAR 재보정과 ROI, 재학습 파이프라인 통합을 차례로 수행한다. 최종 목표는 구간별 소요·지체를 예측해 병목을 사전에 찾고 우회 물량 배분안을 만들되, 우회 명령의 하달 여부는 반드시 사람이 결정하는 수송 소요예측 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(작전군수처, 수송대대)에 즉시 도입 가능한 "완성형 국방 AI 수송 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 기술 명세 및 운영 매뉴얼** | MSR 통과 능력 데이터 명세서(`docs/mov01_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mov01_operations_guide.md`) | 미션 1, 10 |
| **`src/`** | **군수 데이터 정제 및 분석 모듈** | 통과 지체시간 계산 전처리 파이프라인(`src/data_preprocessing.py`), BPR 기반 1차 지체 예측 Baseline(`src/baseline_delay.py`) | 미션 2, 3 |
| **`notebooks/`** | **통계 검증 및 심층 EDA** | MSR 구간 지체 탐색적 데이터 분석 및 파레토 분석(`notebooks/mov01_eda_analysis.ipynb`) | 미션 3 |
| **`src/`** | **AI 예측 & 우회로 탐색 엔진** | Max-Flow 기반 병목 사전 식별 및 우회로 탐색(`src/network_flow.py`), 유류 소모 및 소요시간 산출(`src/route_cost_engine.py`) | 미션 4, 5 |
| **`src/`** | **작전 상황판 & 스트레스 모의** | Streamlit 기반 전술 COP 대시보드(`src/dashboard_app.py`), 3개 축선 봉쇄 스트레스 시험(`src/stress_test.py`) | 미션 6, 8 |
| **`pipeline/`** | **HITL 통제 & 지속적 학습** | 작전군수관 3단계 승인 파이프라인(`pipeline/hitl_workflow.py`), 챔피언/챌린저 모델 자동 재학습 | 미션 7, 10 |
| **`reports/`** | **검증 및 AAR 리포트** | 3개 축선 봉쇄 스트레스 검증 보고서(`reports/mov01_stress_test_report.md`), AAR 오차 분석 및 ROI 보고서(`reports/mov01_roi_analysis.md`) | 미션 8, 9 |
| **`root`** | **통합 배포 패키지** | 통합 실행 진입점(`main.py`), 국방 망분리 Dockerfile 패키지 | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 구간별 통과 용량 기반 1차 수송 지체 예측 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

통과 용량 기반 1차 수송 지체 예측 Baseline 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **구간별 통과 용량 기반 BPR 지체 함수로 1차 수송 지체 예측 Baseline을 만들고, EDA로 Baseline의 한계와 핵심 요인을 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **계획 물량을 공칭 용량으로 나눈 이용률로 구간 지체를 계산하고 통과시간의 오차를 측정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `notebooks/mov01_eda_analysis.ipynb 및 EDA 요약 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 의사결정에 직관적인 인사이트 제공 수준

#### 미션 PASS 조건

- 정량 기준: EDA 시각화 차트 8종 이상 생성, 유의미한 상관관계 피처 5개 이상 도출


- 진행 기준: EDA 노트북 실행 완료 및 핵심 발견점(Key Findings) 문서화


- 검증 방법: 통계적 가설 검정(p-value, 상관계수)을 통한 유의성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""구간별 통과 용량(BPR 함수) 기반 1차 수송 지체 예측 Baseline을 만들어 오차를 측정하고, EDA 차트 8종과 상관 피처·파레토(상위 20% 구간) 분석으로 Baseline의 한계를 찾는다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 노선망·구간 통과량 생성기 (실제 도로·지명·부대 없음)
# 가상 출발지(SRC)에서 가상 목적지(DST)까지 3개 축선(A·B·C)과 연결로 2개, 구간 10개. 150일치 피크시간 통과량·지체 시간을 만든다.
# ---------------------------------------------------------------------------
SEGS = pd.DataFrame([
    ("A1", "SRC", "J1", 12, 130, "일반"), ("A2", "J1", "J2", 8, 70, "교량"), ("A3", "J2", "DST", 10, 140, "교차로"),
    ("B1", "SRC", "J3", 15, 100, "일반"), ("B2", "J3", "J2", 9, 80, "터널"), ("B3", "J3", "DST", 14, 95, "일반"),
    ("C1", "SRC", "J4", 18, 70, "일반"), ("C2", "J4", "DST", 20, 75, "교량"),
    ("X1", "J1", "J3", 7, 50, "교차로"), ("X2", "J4", "J3", 10, 60, "일반")],
    columns=["seg", "u", "v", "km", "cap", "kind"])               # cap = 피크시간 통과 용량(대/시, 교육용 가정값)
KSPD = {"일반": 55, "교량": 40, "터널": 45, "교차로": 45}           # 구간 유형별 자유 주행 속도(km/h, 가정값)
SEGS["t0"] = SEGS["km"] / SEGS["kind"].map(KSPD) * 60              # 자유 주행 시간(분)
UTIL = dict(A1=.78, A2=.85, A3=.66, B1=.70, B2=.72, B3=.64, C1=.55, C2=.58, X1=.45, X2=.40)   # 평시 이용률


def bpr(t0, v, cap, a=0.15, b=4.0):
    """BPR 지체 함수(교통공학의 표준형): 통과량/용량 비가 커질수록 지체가 급증한다. (분)"""
    return t0 * a * (v / cap) ** b


def make_data(seed=11, days=150):
    """구간 x 일자 패널. v=실제 피크 통과량, plan_v=수송계획 수량(긴급 불출 미반영), delay=실제 지체(분)."""
    rng = np.random.default_rng(seed)
    d_idx = np.arange(days)
    lvl = np.array([1.0, 1.08, 1.04, 1.10, 1.02, 0.82, 0.78])[d_idx % 7] * (1 + 0.0012 * d_idx)
    lvl = lvl * np.where((d_idx >= 100) & (d_idx < 110), 1.30, 1.0)       # 수송량 폭증 기간(가상)
    prev, rows = {s: 0.0 for s in SEGS["seg"]}, []
    for d in d_idx:
        rain = bool(rng.random() < 0.18)
        for r in SEGS.itertuples():
            maint, urgent = bool(rng.random() < 0.06), bool(rng.random() < 0.05)   # 비계획 정비 / 긴급 불출
            v = r.cap * UTIL[r.seg] * lvl[d] * (1.25 if urgent else 1.0) * rng.normal(1, 0.05)
            cap_eff = r.cap * (0.65 if maint else 1.0) * (0.92 if rain else 1.0)
            delay = max(r.t0 * 0.2 * (v / cap_eff) ** 3.5 + 0.25 * prev[r.seg] + rng.normal(0, 0.4), 0.0)
            prev[r.seg] = delay
            rows.append(dict(day=int(d), dow=int(d % 7), seg=r.seg, kind=r.kind, km=r.km, cap=r.cap, t0=r.t0,
                             plan_v=v / (1.25 if urgent else 1.0) * rng.normal(1, 0.04), v=v, maint=int(maint), urgent=int(urgent),
                             rain=int(rain), cap_eff=cap_eff, delay=delay))
    df = pd.DataFrame(rows)
    df["transit"] = df["t0"] * (1 + 0.08 * df["rain"]) + df["delay"]
    return df


def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(np.abs(y - p) / np.abs(y)) * 100)


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 지체 예측용 파생 피처 생성(당일 오전 시점에 알 수 있는 정보만 사용: 계획 물량, 공지 정비, 기상 예보, 전일·전주 실적)
# ---------------------------------------------------------------------------
FEATS = ["plan_v", "cap", "cap_eff", "t0", "km", "maint", "rain", "util", "util_eff", "util_eff4", "bpr_base", "lag1_delay", "lag7_delay",
         "roll7_v", "surge_ratio", "is_weekend", "dow_sin", "dow_cos", "k_교량", "k_터널", "k_교차로"]


def add_features(df):
    d = df.sort_values(["seg", "day"]).copy()
    g = d.groupby("seg")
    d["lag1_delay"] = g["delay"].shift(1)
    d["lag7_delay"] = g["delay"].shift(7)
    d["roll7_v"] = g["v"].transform(lambda s: s.shift(1).rolling(7).mean())
    d["util"] = d["plan_v"] / d["cap"]
    d["util_eff"] = d["plan_v"] / d["cap_eff"]
    d["util_eff4"] = d["util_eff"] ** 4
    d["bpr_base"] = bpr(d["t0"], d["plan_v"], d["cap"])
    d["surge_ratio"] = d["plan_v"] / d["roll7_v"]
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2 * np.pi * d["dow"] / 7), np.cos(2 * np.pi * d["dow"] / 7)
    for k in ("교량", "터널", "교차로"):
        d["k_" + k] = (d["kind"] == k).astype(int)
    return d.dropna().sort_values(["day", "seg"]).reset_index(drop=True)
# ---------------------------------------------------------------------------

df = make_data()
d = add_features(df)

# 1) Baseline: 계획 물량 / 공칭 용량으로 BPR 지체 계산(정비·기상·전일 지체 미반영)
d["pred_base"] = d["bpr_base"]
d["pred_transit"] = d["t0"] + d["pred_base"]
print(f"[1] Baseline(BPR) 지체 예측: 통과시간 MAPE {mape(d.transit, d.pred_transit):.1f}%, 통과시간 MAE {mae(d.transit, d.pred_transit):.2f}분, 지체 MAE {mae(d.delay, d.pred_base):.2f}분")
bias = (d["pred_base"] - d["delay"])
grp = pd.DataFrame({"평균 오차(예측-실제, 분)": [bias[d.maint == 1].mean(), bias[d.maint == 0].mean(), bias[d.rain == 1].mean(), bias[d.rain == 0].mean(), bias[d.urgent == 1].mean(), bias[(d.day >= 100) & (d.day < 110)].mean()]},
                   index=["비계획 정비일", "정비 없음", "우천일", "맑음", "긴급 불출일", "수송량 폭증 기간"]).round(2)
print("[2] 조건별 Baseline 오차(음수 = 과소예측)")
print(grp.to_string())
print("    -> Baseline은 정비·긴급 불출·폭증 조건에서 지체를 과소예측한다(한계).")

# 2) 상관 분석: 지체와 각 피처의 스피어만 상관·p-value
rows = []
for f in FEATS:
    r, p = stats.spearmanr(d[f], d["delay"])
    rows.append((f, r, p))
cor = pd.DataFrame(rows, columns=["피처", "상관", "p"]).assign(abs_r=lambda x: x["상관"].abs()).sort_values("abs_r", ascending=False)
sig = cor[(cor["abs_r"] >= 0.3) & (cor["p"] < 0.05)]
print(f"\n[3] 지체와 유의한 상관(|r|>=0.3, p<0.05) 피처 {len(sig)}개 (목표 5개 이상: {'달성' if len(sig) >= 5 else '미달'})")
print(sig.head(7)[["피처", "상관", "p"]].round(3).to_string(index=False))
t, p = stats.ttest_ind(d.loc[d.maint == 1, "delay"], d.loc[d.maint == 0, "delay"], equal_var=False)
print(f"    가설검정: 비계획 정비일 vs 평시 지체 평균차 t={t:.1f}, p={p:.2g}")

# 3) 파레토(ABC): 구간별 총 지체 기여
tot = d.groupby("seg")["delay"].sum().sort_values(ascending=False)
cum = tot.cumsum() / tot.sum()
top = tot.index[:max(1, int(np.ceil(len(tot) * 0.2)))]
print(f"\n[4] 파레토: 상위 20% 구간({', '.join(top)})이 총 지체의 {tot[top].sum() / tot.sum():.0%}를 차지, 상위 50% 구간 {cum.iloc[4]:.0%}")

# 4) EDA 차트 8종
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
daily = d.groupby("day")["v"].sum(); ax[0, 0].plot(daily.index, daily.values); ax[0, 0].set_title("1 Daily total peak volume")
ax[0, 1].hist(d["delay"], bins=40); ax[0, 1].set_title("2 Delay distribution")
ax[0, 2].scatter(d["util"], d["delay"], s=6, alpha=.4); ax[0, 2].set_title("3 Utilization vs delay")
mean_seg = d.groupby("seg")["delay"].mean().sort_values(); ax[0, 3].barh(mean_seg.index, mean_seg.values); ax[0, 3].set_title("4 Mean delay by segment")
ax[1, 0].boxplot([d.loc[d.dow == k, "delay"] for k in range(7)]); ax[1, 0].set_title("5 Delay by weekday")
ax[1, 1].boxplot([d.loc[d.rain == 0, "delay"], d.loc[d.rain == 1, "delay"]])
ax[1, 1].set_xticklabels(["dry", "rain"]); ax[1, 1].set_title("6 Rain effect")
ax[1, 2].boxplot([d.loc[d.maint == 0, "delay"], d.loc[d.maint == 1, "delay"]])
ax[1, 2].set_xticklabels(["normal", "maintenance"]); ax[1, 2].set_title("7 Maintenance effect")
ax[1, 3].bar(range(len(tot)), (tot / tot.sum()).values); ax[1, 3].plot(range(len(tot)), cum.values, "r-o"); ax[1, 3].set_title("8 Pareto of total delay")
plt.suptitle("MOV01 EDA (synthetic data)"); plt.tight_layout(); plt.show()
print("\n[5] EDA 차트 8종 생성 완료(목표 8종 이상: 달성)")
print("핵심 발견: 지체는 이용률의 비선형 함수이며 정비·폭증·긴급 불출이 겹치면 급증한다. 이는 M4 파생 피처 설계의 근거가 된다.")


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[1] Baseline(BPR) 지체 예측: 통과시간 MAPE 7.4%, 통과시간 MAE 1.39분, 지체 MAE 1.23분
[2] 조건별 Baseline 오차(음수 = 과소예측)
           평균 오차(예측-실제, 분)
비계획 정비일              -4.87
정비 없음                -0.96
우천일                  -1.67
맑음                   -1.14
긴급 불출일               -2.62
수송량 폭증 기간            -2.66
    -> Baseline은 정비·긴급 불출·폭증 조건에서 지체를 과소예측한다(한계).

[3] 지체와 유의한 상관(|r|>=0.3, p<0.05) 피처 14개 (목표 5개 이상: 달성)
        피처    상관   p
  util_eff 0.836 0.0
 util_eff4 0.836 0.0
  bpr_base 0.830 0.0
      util 0.793 0.0
lag1_delay 0.730 0.0
    plan_v 0.653 0.0
lag7_delay 0.650 0.0
    가설검정: 비계획 정비일 vs 평시 지체 평균차 t=8.2, p=1.7e-12

[4] 파레토: 상위 20% 구간(A2, A1)이 총 지체의 33%를 차지, 상위 50% 구간 70%

[5] EDA 차트 8종 생성 완료(목표 8종 이상: 달성)
핵심 발견: 지체는 이용률의 비선형 함수이며 정비·폭증·긴급 불출이 겹치면 급증한다. 이는 M4 파생 피처 설계의 근거가 된다.
```
</details>
